In [0]:
%sql
CREATE WIDGET TEXT prm_start_year_month DEFAULT "201001";
CREATE WIDGET TEXT prm_end_year_month DEFAULT "202602";

In [0]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta.tables import DeltaTable
import uuid
import re

In [0]:
start_year_month = dbutils.widgets.get("prm_start_year_month")
end_year_month = dbutils.widgets.get("prm_end_year_month")

In [0]:
%sql
-- 确保进入正确的 Catalog 和 Schema
USE CATALOG nyc;
CREATE SCHEMA IF NOT EXISTS process_silver;

-- 创建目标 Delta 表
CREATE TABLE IF NOT EXISTS process_silver.brz_nyc_taxi_yellow (
    vendor_id STRING,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    passenger_count BIGINT,
    trip_distance DOUBLE,
    rate_code BIGINT,
    store_and_fwd_flag STRING,
    pickup_longitude DOUBLE,
    pickup_latitude DOUBLE,
    dropoff_longitude DOUBLE,
    dropoff_latitude DOUBLE,
    PULocationID BIGINT,
    DOLocationID BIGINT,
    payment_type STRING,
    fare_amount DOUBLE,
    surcharge DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    improvement_surcharge DOUBLE,
    congestion_surcharge DOUBLE,
    airport_fee DOUBLE,
    cbd_congestion_fee DOUBLE,
    total_amount DOUBLE,
    YYYY INT,
    YYYYMM INT, 
    _run_id STRING, 
    _load_timestamp TIMESTAMP, 
    _row_hash STRING, 
    _input_file STRING 
)
USING DELTA
CLUSTER BY (YYYYMM, _input_file, _row_hash)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'comment' = 'NYC Yellow Taxi Silver Table with DQ markers'
);




In [0]:
from pyspark.sql import functions as F 

base_path = "/Volumes/nyc/default/"
yellow_path = base_path + "yellowtaxi/"
delta_target = "nyc.process_silver.delta_yellow_taxi"

In [0]:
import datetime 
import re 

  

def _list_parquet_files(spark, base_path, start_time, end_time, recursive=True): 
    start_val = int(start_time) 
    end_val = int(end_time) 

    # Get the YYYY-MM from file name 
    path_pattern = re.compile(r"yellow_tripdata_(\d{4})-(\d{2})\.parquet") 

    valid_paths = [] 

    # recursive function to get all the files in the directory 
    def _scan(current_path): 
        # use dbutils to list all files in the directory 
        for file_info in dbutils.fs.ls(current_path): 
            if file_info.isDir(): 
                if recursive: 
                    _scan(file_info.path) 

                else: 
                    match = path_pattern.search(file_info.name)
                    if match: 
                        file_date_val = int(match.group(1) + match.group(2)) 

                        if start_val <= file_date_val <= end_val: 
                            valid_paths.append(file_info.path)

    _scan(base_path) 

    return sorted(valid_paths)

In [0]:
# Parameter definition for the bronze table 

_CANONICAL: Tuple[Tuple[str, T.DataType], ...] = (
    ("vendor_id", T.StringType()),
    ("pickup_datetime", T.TimestampType()),
    ("dropoff_datetime", T.TimestampType()),
    ("passenger_count", T.LongType()),
    ("trip_distance", T.DoubleType()),
    ("rate_code", T.LongType()),
    ("store_and_fwd_flag", T.StringType()),
    ("pickup_longitude", T.DoubleType()),
    ("pickup_latitude", T.DoubleType()),
    ("dropoff_longitude", T.DoubleType()),
    ("dropoff_latitude", T.DoubleType()),
    ("PULocationID", T.LongType()),
    ("DOLocationID", T.LongType()),
    ("payment_type", T.StringType()),
    ("fare_amount", T.DoubleType()),
    ("surcharge", T.DoubleType()),
    ("mta_tax", T.DoubleType()),
    ("tip_amount", T.DoubleType()),
    ("tolls_amount", T.DoubleType()),
    ("improvement_surcharge", T.DoubleType()),
    ("congestion_surcharge", T.DoubleType()),
    ("airport_fee", T.DoubleType()),
    ("cbd_congestion_fee", T.DoubleType()),
    ("total_amount", T.DoubleType()),
)

_LINEAGE_COL = "_input_file"

## For Idempotency 
_IDEMPOTENCY_FILE_COL = "_input_file"
_IDEMPOTENCY_HASH_COL = "_row_hash"
_RUN_ID_COL = "_run_id"
_CANONICAL_COLS: Tuple[str, ...] = tuple(name for name, _ in _CANONICAL)

_SOURCE_ROW_INDEX_COL = "_source_row_index"

# Generate the YYYYMM from file name 
_YYYYMM_RE = re.compile(r"yellow_tripdata_(\d{4})-(\d{2})\.parquet$", re.IGNORECASE)


In [0]:
# Define the DQ result class
@dataclass(frozen=True)
class DQResult:
    total_rows: int
    bad_rows: int
    bad_by_rule: Dict[str, int]

In [0]:


class TaxiBronzeLoader:
    def __init__(self, spark):
        self.spark = spark
        self.base_path = "/Volumes/nyc/default/"
        self.yellow_path = base_path + "yellowtaxi/"
        self.delta_target = "nyc.process_silver.brz_nyc_taxi_yellow"


    ## For Idempotency 

    def _stable_string(self, col: F.Column) -> F.Column:
        # Make hashing stable across types + nulls
        return F.coalesce(
            F.when(col.cast("string").isNull(), F.lit("∅")).otherwise(col.cast("string")),
            F.lit("∅"),
        )

    def _add_idempotency_keys(self, df: DataFrame, *, file_col: str = _IDEMPOTENCY_FILE_COL) -> DataFrame:
        cols_for_hash = [F.col(c) for c in _CANONICAL_COLS if c in df.columns]
        if "YYYYMM" in df.columns:
            cols_for_hash.append(F.col("YYYYMM"))
        # If available, include stable per-file row index so identical rows don't collide
        if _SOURCE_ROW_INDEX_COL in df.columns:
            cols_for_hash.append(F.col(_SOURCE_ROW_INDEX_COL))
        hash_input = F.concat_ws("||", *[self._stable_string(c) for c in cols_for_hash])
        return df.withColumn(_IDEMPOTENCY_HASH_COL, F.sha2(hash_input, 256))


    def _merge_dq(self, agg: DQResult, part: DQResult) -> DQResult:
        merged_rules: Dict[str, int] = dict(agg.bad_by_rule)
        for k, v in part.bad_by_rule.items():
            merged_rules[k] = merged_rules.get(k, 0) + v
        return DQResult(
            total_rows=agg.total_rows + part.total_rows,
            bad_rows=agg.bad_rows + part.bad_rows,
            bad_by_rule=merged_rules,
        )


    def _yyyy_yyyymm_from_file_path(self, parquet_path: str, *, fail_on_bad_filename: bool) -> Tuple[Optional[int], Optional[int]]:
        m = _YYYYMM_RE.search(parquet_path)
        if not m:
            if fail_on_bad_filename:
                raise ValueError(
                    f"Bad filename: {parquet_path!r}. Expected suffix like yellow_tripdata_YYYY-MM.parquet"
                )
            return None, None
        y = int(m.group(1))
        mm = int(m.group(2))
        return y, y * 100 + mm
    
    
    def _normalize_one(self, df: DataFrame, *, keep_cols: Tuple[str, ...] = (_LINEAGE_COL,)) -> DataFrame: 
        # Remove "__index_level_0__" 
        if "__index_level_0__" in df.columns:
            df = df.drop("__index_level_0__") 

        cols = set(df.columns) 

        # Map the columns to the canonical columns 
        RENAME_MAP = {
            "VendorID": "vendor_id",
            "tpep_pickup_datetime": "pickup_datetime",
            "tpep_dropoff_datetime": "dropoff_datetime",
            "RatecodeID": "rate_code",
            "extra": "surcharge",
            "Airport_fee": "airport_fee"
        }

        is_new = any(c in cols for c in ["tpep_pickup_datetime", "PULocationID"])

        if is_new: 
            for src, dst in RENAME_MAP.items(): 
                if src in df.columns and dst not in df.columns: 
                    df = df.withColumnRenamed(src, dst) 

        # Normalize the format base on _CANONICAL 
        final_exprs = [] 

        for name, dtype in _CANONICAL: 
            if name in df.columns: 
                # if column exists, normalize 
                if not is_new and name in ("pickup_datetime", "dropoff_datetime"):
                    final_exprs.append(F.to_timestamp(F.col(name)).cast(dtype).alias(name))
                else:
                    final_exprs.append(F.col(name).cast(dtype).alias(name)) 
            else:
                # 如果列不存在，补 NULL 并转型
                final_exprs.append(F.lit(None).cast(dtype).alias(name))

        # Remain Lineage 
        for c in keep_cols: 
            if c in df.columns and c not in [n for n, _ in _CANONICAL]: 
                final_exprs.append(F.col(c))

        return df.select(*final_exprs)
    
    
    def _add_yyyymm_from_path(self, df: DataFrame, path_col: str = _LINEAGE_COL) -> DataFrame:
        # 1. 一次性提取匹配部分 "2026-04"
        # 我们匹配类似 2024-01 这种格式
        date_part = F.regexp_extract(F.col(path_col), r"(\d{4})-(\d{2})", 0)

        # 2. 转换并计算
        # YYYY: 取前 4 位
        # YYYYMM: 去掉中间的横杠直接转 int
        return (
            df.withColumn("YYYY", F.substring(date_part, 1, 4).cast("int"))
            .withColumn("YYYYMM", F.regexp_replace(date_part, "-", "").cast("int"))
        )

    
    def _list_parquet_files(self, spark: SparkSession, base_path:str, start_time:int, end_time:int, *, recursive=True): 
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)

        start_val = int(start_time) 
        end_val = int(end_time) 

        # Get the YYYY-MM from file name 
        path_pattern = re.compile(r"yellow_tripdata_(\d{4})-(\d{2})\.parquet") 

        valid_paths = [] 

        # recursive function to get all the files in the directory 
        def _scan(current_path): 
            # use dbutils to list all files in the directory 
            for file_info in dbutils.fs.ls(current_path): 
                if file_info.isDir(): 
                    if recursive: 
                        _scan(file_info.path) 

                else: 
                    match = path_pattern.search(file_info.name)
                    if match: 
                        file_date_val = int(match.group(1) + match.group(2)) 

                        if start_val <= file_date_val <= end_val: 
                            valid_paths.append(file_info.path)

        _scan(base_path) 

        return sorted(valid_paths)

    def process_single_parquet_file(
        self, spark: SparkSession,
        parquet_path: str,
        *,
        fail_on_bad_filename: bool = False,
        keep_lineage_col: bool = False,
        persist_source_row_index: bool = False,
    ) -> Tuple[DataFrame, DQResult]:
        yyyy, yyyymm = self._yyyy_yyyymm_from_file_path(parquet_path, fail_on_bad_filename=fail_on_bad_filename)
        raw = (
            spark.read.parquet(parquet_path)
            .withColumn(_LINEAGE_COL, F.col("_metadata.file_path"))
            .withColumn(_SOURCE_ROW_INDEX_COL, F.col("_metadata.row_index"))
        )
        norm = self._normalize_one(raw, keep_cols=(_LINEAGE_COL, _SOURCE_ROW_INDEX_COL))
        with_dates = (
            norm.withColumn("YYYY", F.lit(yyyy).cast("int"))
                .withColumn("YYYYMM", F.lit(yyyymm).cast("int"))
        )
        # dq_df, dq = _dq_check(with_dates)
        out = (
            with_dates.withColumnRenamed(_LINEAGE_COL, _IDEMPOTENCY_FILE_COL)
                .withColumn(_RUN_ID_COL, F.expr("uuid()"))
                .withColumn("_load_timestamp", F.current_timestamp())
        )
        out = self._add_idempotency_keys(out, file_col=_IDEMPOTENCY_FILE_COL)
        # `_source_row_index` is only for hashing/uniqueness; don't overload `keep_lineage_col`
        if (not persist_source_row_index) and (_SOURCE_ROW_INDEX_COL in out.columns):
            out = out.drop(_SOURCE_ROW_INDEX_COL)
        if keep_lineage_col and (_LINEAGE_COL not in out.columns):
            out = out.withColumn(_LINEAGE_COL, F.col(_IDEMPOTENCY_FILE_COL))
        
        row_count = with_dates.count() 

        dq = DQResult(
            total_rows=row_count,
            bad_rows=0,
            bad_by_rule={}
        )
        
        return out, dq
        #return out, dq

    def write_yellow_tripdata_to_delta_idempotent(
        self, 
        spark: SparkSession,
        folder: str,
        #delta_target: str,
        start_time: int, 
        end_time: int, 
        *,
        recursive: bool = False,
        fail_on_bad_filename: bool = False,
        keep_lineage_col: bool = False,
        persist_source_row_index: bool = False,
        run_reconciliation: bool = False,
    ) -> DQResult:
        files = self._list_parquet_files(spark, folder, start_time, end_time, recursive=recursive)
        if not files:
            raise FileNotFoundError(f"No .parquet files found under: {folder!r}")
        if not spark.catalog.tableExists(self.delta_target):
            raise ValueError(
                f"Target table {self.delta_target!r} does not exist. Create it first "
                f"(including {_IDEMPOTENCY_FILE_COL} and {_IDEMPOTENCY_HASH_COL})."
            )
        agg = DQResult(total_rows=0, bad_rows=0, bad_by_rule={})
        expected_row_count = 0
        dt = DeltaTable.forName(spark, self.delta_target)
        for path in files:
            _, yyyymm = self._yyyy_yyyymm_from_file_path(path, fail_on_bad_filename=fail_on_bad_filename)
            if yyyymm is None:
                # Should only happen when fail_on_bad_filename=False
                raise ValueError(f"Could not derive YYYYMM from path: {path!r}")
            df, dq_part = self.process_single_parquet_file(
                spark,
                path,
                fail_on_bad_filename=fail_on_bad_filename,
                keep_lineage_col=keep_lineage_col,
                persist_source_row_index=persist_source_row_index,
            )
            merge_cond = (
                f"s.YYYYMM = {yyyymm} AND "
                f"t.YYYYMM = {yyyymm} AND "
                f"t.{_IDEMPOTENCY_FILE_COL} = s.{_IDEMPOTENCY_FILE_COL} AND "
                f"t.{_IDEMPOTENCY_HASH_COL} = s.{_IDEMPOTENCY_HASH_COL}"
            )
            agg = self._merge_dq(agg, dq_part)
            expected_row_count += dq_part.total_rows
            # Critical for rerun performance when target already contains the rows:
            # inserts only; matched rows are left untouched (no rewrite storm)
            
            #(dt.alias("t")
            #.merge(df.alias("s"), merge_cond)
            #.whenNotMatchedInsertAll()
            #.execute())

            partition_cond = f"YYYYMM = {yyyymm}" 


            dt.delete(condition=partition_cond) 

            (df.write.format("delta") \
                .mode("append") \
                .saveAsTable(self.delta_target))
            
            print(f"Merged file: {path.split('/')[-1]} | rows_seen: {dq_part.total_rows}")
        if run_reconciliation:
            print(spark.table(self.delta_target).count())
        return agg


In [0]:
from pyspark.sql import functions as F

BASE_RULES = {
    "missing_pickup_datetime": F.col("pickup_datetime").isNull(),
    "missing_dropoff_datetime": F.col("dropoff_datetime").isNull(),
    "dropoff_before_pickup": (F.col("pickup_datetime").isNotNull())
    & (F.col("dropoff_datetime").isNotNull())
    & (F.col("dropoff_datetime") < F.col("pickup_datetime")),
    "passenger_count_negative": F.col("passenger_count") < 0,
    "trip_distance_too_small": F.col("trip_distance") < 0.1,
    "total_amount_negative": F.col("total_amount") < 0,
    "invalid_YYYYMM": F.col("YYYYMM").isNotNull()
    & ((F.col("YYYYMM") < 190001) | (F.col("YYYYMM") > 300012)),
    "missing_fare_amount": F.col("fare_amount").isNull(),
    "invalid_fare_amount": F.col("fare_amount").isNotNull() & (F.col("fare_amount") < 2.5),
    "invalid_duration_min": (F.col("pickup_datetime").isNotNull())
    & (F.col("dropoff_datetime").isNotNull())
    & ((F.col("duration_min") < 2.0) | (F.col("duration_min") > 300)), 
    "impossible_speed": (F.col("trip_distance") / (F.col("duration_min") / 60) > 100 )
    
}

In [0]:
brz_run = TaxiBronzeLoader(spark)
dq = brz_run.write_yellow_tripdata_to_delta_idempotent(
    spark, folder=yellow_path, 
    start_time = 201001, 
    end_time = 201002, 
    recursive=True, 
    fail_on_bad_filename=True,
)

print(dq)